In [11]:
!pip install -q -U langchain langchain-openai langchain-tavily python-dotenv

In [12]:
from dotenv import load_dotenv
import os
import json

from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_core.tools import tool

load_dotenv()

print("OpenAI:", os.getenv("OPENAI_API_KEY") is not None)
print("Tavily:", os.getenv("TAVILY_API_KEY") is not None)

OpenAI: True
Tavily: True


In [13]:
llm = ChatOpenAI(
    model="gpt-5.6-luna",
    use_responses_api=True
)

print("LLM ready")

LLM ready


In [14]:
with open("thebayrestaurant_jed.json", "r", encoding="utf-8") as f:
    evidence = json.load(f)

evidence_text = json.dumps(
    evidence,
    indent=2,
    ensure_ascii=False
)

print("Research Evidence loaded")
print(evidence_text[:1000])

Research Evidence loaded
{
  "restaurant": {
    "restaurant_id": 3,
    "name": "The Bay Restaurant",
    "instagram_username": "thebayrestaurant_jed",
    "instagram_url": "https://www.instagram.com/thebayrestaurant_jed/",
    "email": null,
    "location": "Jedaah",
    "category": "Cafe, Restaurant, Food & Beverage Company"
  },
  "analysis_metadata": {
    "status": "partial",
    "analyzed_at": "2026-09-14T07:41:51.804323+00:00",
    "posts_scraped": 6,
    "posts_analyzed": 5,
    "posts_failed": 1,
    "source": "instagram",
    "analysis_version": "1.0"
  },
  "profile": {
    "username": "thebayrestaurant_jed",
    "full_name": "•The BAY• 🍃 •ذا باي•",
    "bio": "Refined Indian Cuisine 🇮🇳 \nBold Flavors, Modern Soul.\n📍The bay, Jeddah",
    "followers": 20474,
    "following": 1,
    "posts_count": 432,
    "website": "https://linktr.ee/thebayrestaurant_jed?utm_source=linktree_profile_share&ltsid=e5bd1dd7-599f-4005-84d2-9819fd8d0f9e",
    "verified": false,
    "business_cate

In [15]:
web_search = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced"
)

@tool
def search_instagram_benchmark(query: str) -> str:
    """
    Search the web for current Instagram marketing benchmarks,
    especially for restaurants and food & beverage businesses.
    """
    
    results = web_search.invoke({
        "query": query
    })
    
    return str(results)

print("Benchmark Tool ready")

Benchmark Tool ready


In [16]:
test_result = search_instagram_benchmark.invoke(
    "2026 Instagram posting frequency engagement benchmark for restaurants"
)

print(test_result)

{'query': '2026 Instagram posting frequency engagement benchmark for restaurants', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://evokad.com/restaurant-social-media-marketing-guide-2026', 'title': 'The Restaurant Social Media Marketing Guide 2026', 'content': 'For restaurants specifically, the platform picture gets more nuanced. Hootsuite’s industry benchmarks place dining and hospitality brands at around 3.1% engagement on Instagram, well above the cross-industry average. That premium exists because food content performs natively on visual platforms. What most CMOs miss is how much the numbers shift when follower size is factored in. Accounts with fewer than 10,000 followers average 4.7% engagement on TikTok and 2.8% on Instagram, significantly higher than those with more than 100,000 followers. [...] The highest-converting Instagram content mix for restaurants combines Reels for reach, carousels for depth, and Stories for reservation prompts. 

In [17]:
qualification_prompt = f"""
You are the Qualification & Marketing Gap Analysis Agent for Rawaj.

Your task is to analyze ONE restaurant using its Instagram research evidence
and determine whether it has meaningful marketing gaps.

The restaurant has already passed the ICP criteria.
DO NOT re-evaluate the ICP.

Your role is ONLY to:
- Analyze the restaurant's current Instagram marketing performance.
- Identify marketing gaps supported by evidence.
- Use the Benchmark Search Tool when an external benchmark can meaningfully
  strengthen the evaluation.
- Compare the restaurant's metrics with relevant external benchmarks.
- Explain why each finding represents a marketing gap.
- Prioritize the identified gaps.
- Determine whether the restaurant should be qualified for Rawaj.

DO NOT:
- Generate a marketing strategy.
- Recommend solutions.
- Calculate revenue impact.
- Invent missing data.
- Treat missing information as zero.

====================
RESEARCH EVIDENCE
====================

{evidence_text}

====================
BENCHMARK TOOL
====================

You have access to the search_instagram_benchmark tool.

Use this tool when a measurable Instagram metric can meaningfully be compared
with an external benchmark.

Prioritize:
- Recent benchmarks, preferably 2026.
- Instagram-specific benchmarks.
- Food & Beverage / Restaurant benchmarks when available.
- Credible sources such as established marketing research companies
  and benchmark reports.

When using a benchmark, clearly state:
- Restaurant's actual metric.
- Benchmark value.
- Benchmark context.
- Source/report name.
- Publication year.
- Source URL when available.
- Comparison between the restaurant and benchmark.
- Why the comparison indicates a marketing gap.

A benchmark is a reference point, NOT a universal rule.

If credible sources have different benchmark values, report the difference
and consider the context rather than arbitrarily choosing one.

If no reliable benchmark exists, use the available Instagram evidence and
clearly state that the gap was identified from the restaurant's own evidence.

====================
ANALYSIS
====================

Analyze:

1. RESTAURANT OVERVIEW
- Restaurant name
- Location
- Instagram username
- Followers
- Total posts
- Recent posting activity
- Posting frequency
- Days since last post
- Average likes
- Average comments
- Engagement rate
- Content formats
- Content themes
- CTA usage
- Promotional content
- Menu visibility
- Price visibility
- Offer visibility
- Branding consistency
- Other relevant evidence

Only mention information that is available.

2. MARKETING PERFORMANCE

Evaluate separately:
- Audience engagement
- Posting consistency
- Content quality and variety
- Product/menu visibility
- Promotional communication
- CTA and customer action
- Conversion-oriented information
- Brand presentation
- Audience interaction

For each important finding distinguish:
A. Instagram Evidence
B. External Benchmark Evidence, if used
C. Comparison
D. Marketing Interpretation

3. MARKETING GAPS

Identify ALL meaningful marketing gaps supported by evidence.

For every gap provide:

## Gap: [Gap Name]

Severity: High / Moderate / Low
Priority: 1 = highest priority
Confidence: High / Moderate / Low

### Instagram Evidence
What the research found.

### Evidence Source
Where the evidence came from.

### External Benchmark Evidence
Benchmark value, source/report, year and URL if available.

### Comparison
Restaurant metric versus benchmark.

### Why This Is a Marketing Gap
Explain clearly:

WHAT was found
→ WHERE it came from
→ WHAT the benchmark says, if applicable
→ HOW the restaurant compares
→ WHY this represents a marketing weakness

### Marketing Area
Affected marketing area.

### Customer Journey Stage
Affected customer journey stage.

For non-metric gaps such as menu visibility or CTA usage,
use the actual content analysis as evidence.

Do not create a gap simply because something is different.
There must be reasonable evidence that it represents a marketing weakness.

4. STRENGTHS

Identify important strengths supported by the evidence.

5. DATA LIMITATIONS

Mention limitations such as:
- incomplete post analysis
- unavailable reach
- unavailable impressions
- unavailable saves
- unavailable shares
- unavailable profile visits
- unavailable conversions
- failed analysis
- limited sample size
- benchmark methodology differences

Do not treat missing data as poor performance.

6. FINAL DECISION

Clearly state:

Does this restaurant have meaningful marketing gaps?
YES or NO

Qualification:
Qualified or Unqualified

Explain the decision using only the available evidence.

====================
IMPORTANT RULES
====================

- Use only available restaurant evidence.
- Do not invent metrics or sources.
- Do not assume missing information is zero.
- Do not confuse absence of evidence with evidence of absence.
- Do not generate strategies or solutions.
- Do not calculate revenue impact.
- Prefer recent and relevant benchmarks.
- Never present a benchmark as a universal rule.
- Always name the source when using an external benchmark.
- Never invent a benchmark value or URL.
- If evidence is insufficient, explicitly say so.
- Every identified gap must have a clear evidence-based explanation.

Return a detailed plain-text report with clear headings and bullets.
Do NOT return JSON, Python objects, dictionaries, or code.
"""

In [18]:
from langchain.agents import create_agent

tools = [search_instagram_benchmark]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=qualification_prompt
)

print("Agent ready")

Agent ready


In [19]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": qualification_prompt
        }
    ]
})



In [20]:
for i, message in enumerate(result["messages"]):
    print(f"\n===== MESSAGE {i} =====")
    print("TYPE:", type(message).__name__)

    if type(message).__name__ == "AIMessage":
        print("CONTENT:")
        print(message.content)

        print("\nTEXT:")
        print(message.text)

    elif type(message).__name__ == "ToolMessage":
        print("TOOL:", message.name)


===== MESSAGE 0 =====
TYPE: HumanMessage

===== MESSAGE 1 =====
TYPE: AIMessage
CONTENT:
[{'id': 'rs_0f2a3571e1a005b8006aa9bb16119887d2b624e626c7ae2bb7', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqqbsXKfJU75PD8Mc-_1HbEMrj-B9u3fRBaBSB5_ZKkM6TLgmURXkOuOOyVw1_8BvG-5wlijcFO5a6lPpHR5FSUMXYSYBWPhhyFzAq9f5leMuAjnJVDqL-pNwnoTgbWNx7u_XB6IAHUr5poefAtxI805kextILbwetX3PQ13N1sO9wc30Z2QEhv2bYxls1ImFE5NZKn_mwtxiJdcQ8APV_WQ9pUHkJAyN7VXP24fEE1tydHoAOICOgEhEUI3L6LA0qvy7kyrHLAYdU1kPIm_8JBPCzNqzy3h1TiQzP4DWP8Qc9rqslm8wbCPXHeI1rE2X1XsvGUc_wIpZyoRIfdC65lIqEIC7-5p294Yve1Au4GfsXnSgldYdQeEtxuMfg_eB2FiQcSFve-Q8KOD56TP_DqduvZrm3X2iUwuo3FwgYdtmyb790MrmqMA2Pbq8JBXrsnZDvKRIUQ3ZZl8CRx9X2brKe2QfUzrdrlsYHGci9vI4OKwEFuG0E5f36hAWG3z-gAnjAzNSGv5TuKzWZUV4qjaDmmV8k2fh-9T3Z9yigKwXtgWGR__eeC7HpOd07b3LD_bxPqN3bZwPNDosNjY9l7-6R7feeZAVrxdUKEU0eoxomBtunYAmqriyJKLioKXxmeqZLV8o8pbjXL-JYQp3kV-QH3OoCSaWrYHyDPjcgPL2tIRt5EdCPP56jUdbL83zKaQtvS7wMHW1m7bASL5o2RH2RN6SSPszrjqpu_Rus6s2p2HkSiX7zosDCMKBu

In [21]:
import json

# Get the final AI message
final_message = result["messages"][-1]

# Extract the final text from the Responses API output
final_report = ""

for item in final_message.content:
    if isinstance(item, dict) and item.get("type") == "text":
        final_report = item.get("text", "")
        break

if not final_report:
    raise ValueError("Final report text was not found.")

# Create a clean JSON structure
qualification_output = {
    "restaurant": evidence.get("name"),
    "restaurant_id": evidence.get("restaurant_id"),
    "agent": "Qualification & Marketing Gap Analysis Agent",
    "qualification_report": final_report
}

# Save as JSON
with open("thebayrestaurant_qualification.json", "w", encoding="utf-8") as f:
    json.dump(
        qualification_output,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Qualification report saved successfully.")
print("File: thebayrestaurant_qualification.json")

Qualification report saved successfully.
File: thebayrestaurant_qualification.json


# if any one did not like lines of reading this is the summery from the agent hahahah 

In [23]:
def summarize_qualification_report(report: str) -> str:
    summary_prompt = f"""
Summarize this Qualification & Marketing Gap Analysis report
for the Rawaj project team.

Include:
- Qualification result
- Main marketing gaps
- Important supporting metrics
- Key strengths
- Important data limitations

Keep it concise and easy to read.
Do not provide solutions or strategies.
Do not add information that is not in the report.

REPORT:
{report}
"""

    response = llm.invoke(summary_prompt)
    return response.text


summary = summarize_qualification_report(final_report)

print(summary)

# Rawaj Qualification Summary: The Bay Restaurant

## Qualification Result
**Qualified for Rawaj.**

The restaurant has a sizable audience, clear branding and several evidence-based Instagram marketing gaps, particularly in engagement and conversion-oriented communication.

## Main Marketing Gaps
- **Very low engagement:** Reported engagement is **0.031%**, with **6.33 average likes** and **zero average comments**.
- **Weak recent posting continuity:** **4 posts in 30 days**, averaging **0.93 posts per week**, with a **10-day gap** before analysis.
- **Limited menu and price visibility:** Menu details appeared in only **20%** of analyzed posts; **no prices** were visible.
- **Inconsistent CTAs:** Only **40%** of successfully analyzed posts included a detectable call to action.
- **Promotions lack practical details:** Promotional content represented **60%** of the sample, but product, price and ordering information were not consistently included.
- **Limited observable community interac